# MalthusJAX Level 1: Fitness Evaluators Demo

This notebook demonstrates the **fitness evaluator architecture** in MalthusJAX:
- `BaseEvaluator` — abstract base with `evaluate()` and `evaluate_population()`
- **Binary evaluators**: `BinarySumEvaluator`, `KnapsackEvaluator`
- **Real evaluators**: `SphereEvaluator`, `GriewankEvaluator`, `BoxEvaluator`
- **BBOB benchmark**: `BBOBEvaluator` (wraps evosax)

Key concepts covered:
1. Evaluator configuration and instantiation
2. Single genome vs. population evaluation
3. Maximization vs. minimization
4. Creating random problem instances
5. JAX integration (`jax.jit`, `jax.vmap`)

In [1]:
# Core imports
import jax
import jax.numpy as jnp
import jax.random as jr

# MalthusJAX fitness evaluator imports
from malthusjax.core.fitness import (
    # Binary evaluators
    BinarySumConfig,
    BinarySumEvaluator,
    BoxEvaluator,
    GriewankConfig,
    GriewankEvaluator,
    KnapsackEvaluator,
    # Real evaluators
    SphereConfig,
    SphereEvaluator,
)

# MalthusJAX genome imports
from malthusjax.core.genome import (
    BinaryGenomeConfig,
    BinaryPopulation,
    RealGenome,
    RealGenomeConfig,
    RealPopulation,
)

# Reproducibility
key = jr.PRNGKey(42)
print(f"JAX version: {jax.__version__}")
print(f"Devices: {jax.devices()}")

JAX version: 0.8.0
Devices: [CpuDevice(id=0)]


---
## 1. Binary Sum (OneMax) Evaluator

The simplest binary fitness function: count the number of 1s in the bit-string.
- **Maximize**: More 1s = higher fitness
- **Minimize**: Fewer 1s = higher fitness (counts 0s)

In [2]:
# Create a binary population
binary_config = BinaryGenomeConfig(shape=(20,), p=0.5)
key, subkey = jr.split(key)
binary_pop = BinaryPopulation.init_random(subkey, binary_config, size=10)

print(f"Population size: {len(binary_pop)}")
print(f"Genome length: {binary_config.resolved_shape[0]}")

Population size: 10
Genome length: 20


In [3]:
# Create OneMax evaluator (maximize = count 1s)
onemax_config = BinarySumConfig(maximize=True)
onemax_evaluator = BinarySumEvaluator(config=onemax_config, data=None)

# Evaluate a single genome
single_genome = binary_pop[0]
single_fitness = onemax_evaluator.evaluate(single_genome)
print(f"Single genome: {single_genome.values}")
print(f"OneMax fitness (count of 1s): {single_fitness}")

Single genome: [1 0 0 1 1 1 0 0 0 0 0 0 1 0 0 1 1 0 1 0]
OneMax fitness (count of 1s): 8


In [4]:
# Evaluate entire population
evaluated_pop = onemax_evaluator.evaluate_population(binary_pop)

print(f"Population fitness values: {evaluated_pop.fitness}")
print(f"Best fitness: {jnp.max(evaluated_pop.fitness)} / {binary_config.resolved_shape[0]}")
print(f"Best individual index: {jnp.argmax(evaluated_pop.fitness)}")

Population fitness values: [ 8 14  8  9  8 14 11 12 10 15]
Best fitness: 15 / 20
Best individual index: 9


In [5]:
# ZeroMax (minimize = count 0s)
zeromax_config = BinarySumConfig(maximize=False)
zeromax_evaluator = BinarySumEvaluator(config=zeromax_config, data=None)

zeromax_pop = zeromax_evaluator.evaluate_population(binary_pop)
print(f"ZeroMax fitness (count of 0s): {zeromax_pop.fitness}")

ZeroMax fitness (count of 0s): [12  6 12 11 12  6  9  8 10  5]


---
## 2. Knapsack Evaluator

The classic 0/1 Knapsack problem:
- Each bit represents whether to include an item
- Goal: Maximize total value while respecting weight capacity
- Uses penalty for constraint violations

In [6]:
# Create a random knapsack problem
N_ITEMS = 20
key, subkey = jr.split(key)
knapsack_config = KnapsackEvaluator.create_random_problem(
    subkey, n_items=N_ITEMS, capacity_ratio=0.5, maximize=True
)

print(f"Number of items: {N_ITEMS}")
print(f"Weights (first 5): {knapsack_config.weights[:5]}")
print(f"Values (first 5): {knapsack_config.values[:5]}")
print(f"Capacity: {knapsack_config.capacity:.2f}")
print(f"Total weight if all selected: {jnp.sum(knapsack_config.weights):.2f}")

Number of items: 20
Weights (first 5): [ 6.6149335  3.8637934  5.815345  16.264952  18.477924 ]
Values (first 5): [20.57296    4.0166693 17.171486  42.126053  40.928677 ]
Capacity: 110.79
Total weight if all selected: 221.58


In [7]:
# Create evaluator and evaluate population
knapsack_evaluator = KnapsackEvaluator(config=knapsack_config, data=None)

# Create population matching item count
knapsack_genome_config = BinaryGenomeConfig(shape=(N_ITEMS,), p=0.3)  # sparse selection
key, subkey = jr.split(key)
knapsack_pop = BinaryPopulation.init_random(subkey, knapsack_genome_config, size=50)

# Evaluate
evaluated_knapsack = knapsack_evaluator.evaluate_population(knapsack_pop)

best_idx = jnp.argmax(evaluated_knapsack.fitness)
best_genome = evaluated_knapsack[int(best_idx)]
best_fitness = evaluated_knapsack.fitness[best_idx]

print(f"Best fitness: {best_fitness:.2f}")
print(f"Best selection: {best_genome.values}")
print(f"Items selected: {jnp.sum(best_genome.values)}")

Best fitness: 301.73
Best selection: [0 0 1 1 0 0 1 1 0 0 1 0 0 1 1 0 0 0 1 1]
Items selected: 9


In [8]:
# Analyze best solution
best_weight = jnp.sum(best_genome.values * knapsack_config.weights)
best_value = jnp.sum(best_genome.values * knapsack_config.values)

print("Best solution analysis:")
print(f"  Total weight: {best_weight:.2f} / {knapsack_config.capacity:.2f}")
print(f"  Total value: {best_value:.2f}")
print(f"  Feasible: {best_weight <= knapsack_config.capacity}")

Best solution analysis:
  Total weight: 88.68 / 110.79
  Total value: 301.73
  Feasible: True


---
## 3. Sphere Evaluator

The Sphere function: $f(\mathbf{x}) = \sum_{i=1}^n x_i^2$
- Global minimum at origin: $f(\mathbf{0}) = 0$
- Unimodal, separable, convex
- Standard benchmark for continuous optimization

In [9]:
# Create real population
real_config = RealGenomeConfig(shape=(10,), bounds=(-5.0, 5.0))
key, subkey = jr.split(key)
real_pop = RealPopulation.init_random(subkey, real_config, size=100)

print(f"Population size: {len(real_pop)}")
print(f"Dimensions: {real_config.shape[0]}")
print(f"Bounds: {real_config.bounds}")

Population size: 100
Dimensions: 10
Bounds: (-5.0, 5.0)


In [10]:
# Sphere evaluator (minimization → maximize=False, fitness negated internally)
sphere_config = SphereConfig(maximize=False)
sphere_evaluator = SphereEvaluator(config=sphere_config, data=None)

# Evaluate population
evaluated_sphere = sphere_evaluator.evaluate_population(real_pop)

# For minimization, "best" is the maximum of negated values (closest to 0)
best_idx = jnp.argmax(evaluated_sphere.fitness)
best_fitness = evaluated_sphere.fitness[best_idx]
best_genome = evaluated_sphere[int(best_idx)]

print(f"Fitness range: [{jnp.min(evaluated_sphere.fitness):.4f}, {jnp.max(evaluated_sphere.fitness):.4f}]")
print(f"Best fitness (negated sphere): {best_fitness:.4f}")
print(f"Best genome values: {best_genome.values}")
print(f"Actual sphere value: {-best_fitness:.4f}")

Fitness range: [-122.9869, -34.1432]
Best fitness (negated sphere): -34.1432
Best genome values: [-0.02340794  0.8792865   1.0067439  -2.247678    0.15223503 -2.8644228
  0.7400942  -2.981596    3.0739117   0.43502212]
Actual sphere value: 34.1432


In [11]:
# Verify with manual calculation
manual_sphere = jnp.sum(jnp.square(best_genome.values))
print(f"Manual sphere calculation: {manual_sphere:.4f}")
print(f"Matches negated fitness: {jnp.isclose(manual_sphere, -best_fitness)}")

Manual sphere calculation: 34.1432
Matches negated fitness: True


---
## 4. Griewank Evaluator

The Griewank function: $f(\mathbf{x}) = 1 + \frac{1}{4000}\sum_{i=1}^n x_i^2 - \prod_{i=1}^n \cos\left(\frac{x_i}{\sqrt{i}}\right)$
- Global minimum at origin: $f(\mathbf{0}) = 0$
- Multimodal with many local minima
- More challenging than Sphere

In [12]:
# Griewank evaluator
griewank_config = GriewankConfig(maximize=False)
griewank_evaluator = GriewankEvaluator(config=griewank_config, data=None)

# Evaluate same population
evaluated_griewank = griewank_evaluator.evaluate_population(real_pop)

best_idx = jnp.argmax(evaluated_griewank.fitness)
best_fitness = evaluated_griewank.fitness[best_idx]

print(f"Griewank fitness range: [{jnp.min(evaluated_griewank.fitness):.4f}, {jnp.max(evaluated_griewank.fitness):.4f}]")
print(f"Best Griewank fitness (negated): {best_fitness:.4f}")
print(f"Actual Griewank value: {-best_fitness:.4f}")

Griewank fitness range: [-1.1362, -0.7864]
Best Griewank fitness (negated): -0.7864
Actual Griewank value: 0.7864


In [13]:
# Compare Sphere vs Griewank rankings
sphere_ranks = jnp.argsort(jnp.argsort(-evaluated_sphere.fitness))  # Higher = better
griewank_ranks = jnp.argsort(jnp.argsort(-evaluated_griewank.fitness))

rank_correlation = jnp.corrcoef(sphere_ranks, griewank_ranks)[0, 1]
print(f"Rank correlation (Sphere vs Griewank): {rank_correlation:.4f}")

Rank correlation (Sphere vs Griewank): 0.6464


---
## 5. Box-Constrained Evaluator

Optimization with explicit box constraints:
- Target point to reach
- Constraint violations are penalized
- Good for constrained optimization problems

In [14]:
# Create random box-constrained problem
key, subkey = jr.split(key)
box_config = BoxEvaluator.create_random_problem(
    subkey, dimensions=10, box_size=10.0, maximize=False
)

print(f"Target point: {box_config.target_point}")
print(f"Lower bounds: {box_config.box_bounds[0]}")
print(f"Upper bounds: {box_config.box_bounds[1]}")

Target point: [-2.699635   1.1221957 -1.3912237  3.3505142  1.2461019  1.8103015
 -1.0696805  2.2742653 -4.496119   3.4059799]
Lower bounds: [-5.199635   -1.3778043  -3.8912237   0.8505142  -1.2538981  -0.68969846
 -3.5696805  -0.22573471 -6.996119    0.9059799 ]
Upper bounds: [-0.19963503  3.6221957   1.1087763   5.8505144   3.7461019   4.310302
  1.4303195   4.7742653  -1.996119    5.90598   ]


In [15]:
# Box evaluator
box_evaluator = BoxEvaluator(config=box_config, data=None)

# Create population in wider search space
wide_config = RealGenomeConfig(shape=(10,), bounds=(-10.0, 10.0))
key, subkey = jr.split(key)
wide_pop = RealPopulation.init_random(subkey, wide_config, size=100)

# Evaluate
evaluated_box = box_evaluator.evaluate_population(wide_pop)

best_idx = jnp.argmax(evaluated_box.fitness)
best_fitness = evaluated_box.fitness[best_idx]
best_genome = evaluated_box[int(best_idx)]

print(f"Best box fitness: {best_fitness:.4f}")
print(f"Best genome: {best_genome.values}")

Best box fitness: -13468.1562
Best genome: [-3.7861085   1.5228701   0.99054575 -2.6523685   3.4546876  -6.2627554
 -3.1801248   1.3997269   2.3832178   3.9021826 ]


In [16]:
# Check constraint satisfaction
lower, upper = box_config.box_bounds
in_bounds = jnp.all((best_genome.values >= lower) & (best_genome.values <= upper))
distance_to_target = jnp.sqrt(jnp.sum(jnp.square(best_genome.values - box_config.target_point)))

print(f"Feasible (in bounds): {in_bounds}")
print(f"Distance to target: {distance_to_target:.4f}")

Feasible (in bounds): False
Distance to target: 12.8799


---
## 6. BBOB Benchmark Evaluator (evosax wrapper)

The Black-Box Optimization Benchmark (BBOB) suite provides 24 standard test functions.
MalthusJAX wraps evosax's implementation for seamless integration.

**Note**: Requires `evosax` package to be installed.

In [17]:
# Import BBOB evaluator (may require evosax)
try:
    from malthusjax.core.fitness.bbob_evaluator import BBOBConfig, BBOBEvaluator
    BBOB_AVAILABLE = True
    print("BBOB evaluator available!")
except ImportError as e:
    BBOB_AVAILABLE = False
    print(f"BBOB not available: {e}")
    print("Install evosax: pip install evosax")

BBOB evaluator available!


In [18]:
if BBOB_AVAILABLE:
    # Create BBOB evaluator for Rastrigin function
    bbob_config = BBOBConfig(
        maximize=False,
        fn_name="rastrigin",
        num_dims=10,
        seed=42,
    )
    bbob_evaluator = BBOBEvaluator.create(bbob_config)

    # Evaluate population
    bbob_pop = RealPopulation.init_random(jr.PRNGKey(999), real_config, size=100)
    evaluated_bbob = bbob_evaluator.evaluate_population(bbob_pop)

    print(f"BBOB Rastrigin fitness range: [{jnp.min(evaluated_bbob.fitness):.4f}, {jnp.max(evaluated_bbob.fitness):.4f}]")
    print(f"Best fitness: {jnp.max(evaluated_bbob.fitness):.4f}")
else:
    print("Skipping BBOB demo (evosax not installed)")

BBOB Rastrigin fitness range: [37.9050, 4994.6372]
Best fitness: 4994.6372


---
## 7. JIT Compilation

Evaluators are JAX PyTrees and fully compatible with `jax.jit`.

In [19]:
# JIT compile the evaluate method
jit_evaluate = jax.jit(sphere_evaluator.evaluate)

# Warm-up (compilation)
test_genome = real_pop[0]
_ = jit_evaluate(test_genome)

# Benchmark
import time

# Non-JIT
start = time.perf_counter()
for _ in range(1000):
    _ = sphere_evaluator.evaluate(test_genome)
non_jit_time = time.perf_counter() - start

# JIT
start = time.perf_counter()
for _ in range(1000):
    _ = jit_evaluate(test_genome)
jit_time = time.perf_counter() - start

print(f"Non-JIT time (1000 evals): {non_jit_time*1000:.2f} ms")
print(f"JIT time (1000 evals): {jit_time*1000:.2f} ms")
print(f"Speedup: {non_jit_time/jit_time:.1f}x")

Non-JIT time (1000 evals): 42.05 ms
JIT time (1000 evals): 3.68 ms
Speedup: 11.4x


In [20]:
# JIT compile population evaluation
jit_evaluate_pop = jax.jit(sphere_evaluator.evaluate_population)

# Larger population for benchmarking
large_pop = RealPopulation.init_random(jr.PRNGKey(123), real_config, size=10000)

# Warm-up
_ = jit_evaluate_pop(large_pop)

# Benchmark
start = time.perf_counter()
for _ in range(100):
    _ = jit_evaluate_pop(large_pop)
jit_pop_time = time.perf_counter() - start

print(f"JIT population eval (10K individuals, 100 iterations): {jit_pop_time*1000:.2f} ms")
print(f"Average per iteration: {jit_pop_time*10:.4f} ms")

JIT population eval (10K individuals, 100 iterations): 1.75 ms
Average per iteration: 0.0175 ms


---
## 8. Custom Evaluator Pattern

Creating your own fitness evaluator following the MalthusJAX pattern.

In [21]:
from typing import Any

from flax import struct

from malthusjax.core.fitness.base import BaseEvaluator, BaseEvaluatorConfig


@struct.dataclass
class RosenbrockConfig(BaseEvaluatorConfig):
    """Configuration for Rosenbrock function."""
    a: float = 1.0
    b: float = 100.0

@struct.dataclass
class RosenbrockEvaluator(BaseEvaluator[RealGenome, RosenbrockConfig, Any]):
    """Rosenbrock function: f(x,y) = (a-x)² + b(y-x²)²
    
    Generalized to n dimensions as sum of consecutive pairs.
    Global minimum at (a, a², a³, ...) with f = 0.
    """
    config: RosenbrockConfig
    data: Any = struct.field(pytree_node=False, default=None)

    def evaluate(self, genome: RealGenome) -> jnp.ndarray:
        x = genome.values
        a, b = self.config.a, self.config.b

        # Sum over consecutive pairs
        rosenbrock = jnp.sum(
            (a - x[:-1])**2 + b * (x[1:] - x[:-1]**2)**2
        )

        # Return negated for minimization (higher = better)
        return jax.lax.select(self.config.maximize, rosenbrock, -rosenbrock)

In [22]:
# Use custom evaluator
rosenbrock_config = RosenbrockConfig(maximize=False, a=1.0, b=100.0)
rosenbrock_evaluator = RosenbrockEvaluator(config=rosenbrock_config, data=None)

# Evaluate population
evaluated_rosenbrock = rosenbrock_evaluator.evaluate_population(real_pop)

best_idx = jnp.argmax(evaluated_rosenbrock.fitness)
best_fitness = evaluated_rosenbrock.fitness[best_idx]
best_genome = evaluated_rosenbrock[int(best_idx)]

print(f"Rosenbrock fitness range: [{jnp.min(evaluated_rosenbrock.fitness):.4f}, {jnp.max(evaluated_rosenbrock.fitness):.4f}]")
print(f"Best fitness (negated): {best_fitness:.4f}")
print(f"Best genome (first 5): {best_genome.values[:5]}")
print("Optimal point: [1, 1, 1, ...]")

Rosenbrock fitness range: [-262693.8125, -15169.1797]
Best fitness (negated): -15169.1797
Best genome (first 5): [ 2.341863   -1.3936853  -0.08241653 -0.34116983  2.803936  ]
Optimal point: [1, 1, 1, ...]


---
## Summary

| Evaluator | Problem Type | Key Config | Notes |
|-----------|-------------|------------|-------|
| `BinarySumEvaluator` | OneMax/ZeroMax | `maximize` | Count 1s or 0s |
| `KnapsackEvaluator` | 0/1 Knapsack | `weights`, `values`, `capacity` | Penalty for violations |
| `SphereEvaluator` | Continuous | `maximize` | $\sum x_i^2$, min at origin |
| `GriewankEvaluator` | Continuous | `maximize` | Multimodal benchmark |
| `BoxEvaluator` | Constrained | `target_point`, `box_bounds` | Penalty constraints |
| `BBOBEvaluator` | BBOB Suite | `fn_name`, `num_dims` | 24 standard benchmarks |

All evaluators:
- Use `@struct.dataclass` for JAX compatibility
- Implement `evaluate(genome)` for single individuals
- Inherit `evaluate_population()` from `BaseEvaluator` (uses `jax.vmap`)
- Support `maximize` flag for optimization direction